# 12F — Physics-informed shared movement encoder (exploratory, compute-heavy, may be deferred)

**Part 1 · Research question (Q6).** Can an encoder trained on BOTH Combine drills and NFL movement episodes learn a representation that preserves interpretable physics and transfers across domains better than engineered features and the Combine-only TCN-B?

**Part 2 · Declared hypothesis.** *Primary:* the INDUCTIVE F4 encoder (masked reconstruction + physics heads + temporal contrastive + matched-speed-state cross-domain alignment) beats the random encoder AND the engineered features (F0) on held-out cross-domain athlete retrieval, and M1 + its fold embeddings lowers held-out RMSE of the 12C primary target vs M1. *Secondary:* F2 / F3 / F5 / F6 and the ablations; learning curve against the amount of unlabeled NFL data.

**Two regimes, two tables, never one leaderboard.** TRANSDUCTIVE = trained on every sequence (no identity, no NFL outcome) and used only for label-free diagnostics. INDUCTIVE = one encoder per outer fold trained without the held-out athletes' sequences of either domain — the only regime that enters NFL prediction.

**Negative controls.** Random frozen encoder · shuffled athlete identity (retrieval null) · shuffled physics targets (F4_shuffled_phys) · no physics loss (F4_nophys) · domain-only classifier AUC (high = domains separable; NOT a success).

**Engineering.** One resumable tracking scan for NFL windows (checkpoint shards per season × batch); an immutable memory-mapped sequence store; epoch checkpoints as immutable shards (no overwrite — the Windows replace problem of earlier milestones cannot recur); wall-clock cap per encoder; AMP on CUDA, CPU fallback; progress and ETA. If PyTorch is absent the notebook records `not_feasible` and stops — 12G proceeds without it.

**Prerequisites.** 12.0; 12C for the NFL transfer section.

In [ ]:
RUN_MODE = 'full'
SEED = 20261015
FORCE_REBUILD = False
SAVE_TABLES = True
SAVE_FIGURES = True
NUM_WORKERS = 0
MEMORY_BUDGET_GB = 6.0
TRAIN_TRANSDUCTIVE = True
RUN_ABLATIONS = True        # six extra transductive encoders (incl. the shuffled-physics control)
TRAIN_INDUCTIVE = True      # F2 and F4 × outer folds
LEARNING_CURVE = True       # F4 transductive with 10 % and 30 % of the NFL windows
DEVICE = None               # None = cuda when available, else cpu

In [ ]:
from nflbdb.data.config import Phase01Config, configure_runtime

cfg = Phase01Config.load()
configure_runtime(cfg)

import numpy as np
import polars as pl
from IPython.display import Markdown, display

from nflbdb.research.advanced_movement import config as amc
from nflbdb.research.advanced_movement.artifacts import ResearchStore
from nflbdb.research.advanced_movement.m11 import config as m11c
from nflbdb.research.advanced_movement.m11 import nfl_physics as nph
from nflbdb.research.advanced_movement.m11 import representation as rep
from nflbdb.research.advanced_movement.m11 import sources as src
from nflbdb.research.advanced_movement.m11 import speed_state as ss
from nflbdb.research.advanced_movement.m12 import advanced_evaluation as ae
from nflbdb.research.advanced_movement.m12 import config as m12c
from nflbdb.research.advanced_movement.m12 import figures as fg
from nflbdb.research.advanced_movement.m12 import foundation as fd
from nflbdb.research.advanced_movement.m12 import reporting as rp
from nflbdb.research.advanced_movement.m12 import shared_movement_encoder as se
from nflbdb.research.advanced_movement.m12 import synthesis as sy
from nflbdb.research.advanced_movement.m12 import validation as v12

controls = amc.NotebookControls(run_mode=RUN_MODE, force_rebuild=FORCE_REBUILD, save_figures=SAVE_FIGURES, save_tables=SAVE_TABLES,
                                seed=SEED, num_workers=NUM_WORKERS).validate()
store = ResearchStore(cfg, controls)
amc.seed_everything(controls.seed)
C = m12c.M12Config().for_mode(controls)
C11 = m11c.M11Config().for_mode(controls)
S = src.M11Sources(cfg, store, C11)
D = fd.M12Data(store, C, S)
PROV = m12c.provenance('12F', controls, C)
W = store.write_frame
MAN = fd.check_foundation(store, C)
if not se.TORCH:
    rp.run_manifest(store, '12F', PROV, [], m12c.NOT_FEASIBLE, {'reason': 'PyTorch not importable'})
    raise RuntimeError('12F is not feasible here: PyTorch is not installed. The manifest records not_feasible; run 12G without 12F.')
import torch
DEV = DEVICE or ('cuda' if torch.cuda.is_available() else 'cpu')
label = 'SMOKE — wiring check, numbers are not results' if controls.smoke else 'FULL'
display(Markdown(f'**Run mode: `{controls.run_mode}`** ({label}) · config `{C.hash()}` · device `{DEV}`'))
checks = v12.run_all((*v12.CHECKS_GENERAL, *v12.CHECKS_ENCODER))
display(checks)
v12.assert_all_passed(checks)

## Part 3 · Data — unified sequence store (built once, immutable)

Combine: every tracked attempt (all drill families; longest valid segment from 0.5 s before onset, ≤ 6.4 s). NFL: a deterministic capped sample of 09A episodes (≤ `nfl_windows_per_athlete_demand` per athlete × demand, ≤ `nfl_windows_max` overall; per-athlete caps = athlete-grouped balance), 4.8 s windows from one resumable tracking scan. Channels in FIXED physical units (no athlete statistic), masks for invalid / padded frames — no missing signal is fabricated (heading-dependent channels are zero AND flagged invalid).

In [ ]:
off = D.officials()
ep = D.episodes()
POS = dict(zip(ep['nfl_id'].to_list(), ep['position_group'].to_list())) | {int(i): p for i, p in zip(off['nfl_id'].to_list(), off['position_group'].to_list()) if p is not None}
key_file = store.data_dir / 'm12f_sequence_store.json'
st_ = None
if key_file.is_file() and not FORCE_REBUILD:
    k_ = store.read_json('m12f_sequence_store', fd.NB['12F'])['key']
    cand = rep.SequenceStore(store.sequences_dir, k_)
    if (cand.dir / 'COMPLETE.json').is_file():
        st_ = cand
if st_ is None:
    sample = nph.sample_episodes(ep, se.m11_window_config(C), C.seed)
    display(ae.memory_plan([{'item': 'NFL window frames (sample × 48)', 'n_rows': sample.height * C.seq_len, 'n_cols': 12, 'bytes_per_value': 4},
                            {'item': 'sequence store X (N × 64 × 6 float32, memory-mapped)', 'n_rows': sample.height + 7000, 'n_cols': se.STORE_LEN * 6, 'bytes_per_value': 4}], MEMORY_BUDGET_GB))
    windows = nph.NflWindowExtractor(cfg, se.m11_window_config(C), S.oc, store.shards_dir, controls.run_mode, limit_games=C11.smoke_games if controls.smoke else None).compute(S.plays(), sample)
    frames = [S.combine_frames([f]) for f in src.FAMILY_ORDER]
    st_ = se.build_store(store.sequences_dir, frames, windows, POS)
    del frames, windows
    if controls.save_tables:
        store.write_json('m12f_sequence_store', {'key': st_.dir.name, 'dir': str(st_.dir)})
X, M, meta = st_.open()
meta = meta.with_columns(pl.col('nfl_id').replace_strict(POS, default=None, return_dtype=pl.String).alias('position_group')) if 'position_group' not in meta.columns else meta
P = se.physics_targets(X, M)
STATE, CAT = se.state_ids(X, M), se.category_ids(meta)
DOM = (meta['domain'] == 'nfl').to_numpy().astype(np.int64)
PHYS = rep.physical_scalars(X, M)
fmap = dict(zip(*D.m12('m12_fold_manifest').select('nfl_id', 'fold').to_dict(as_series=False).values()))
FOLD = {int(i): (fmap[int(i)] if fmap.get(int(i)) is not None else int(i) % C.outer_folds) for i in meta['nfl_id'].unique().to_list()}
print(f"{X.shape[0]:,} sequences ({int((DOM == 0).sum()):,} Combine, {int(DOM.sum()):,} NFL) × {X.shape[1]} frames × {X.shape[2]} channels · {meta['nfl_id'].n_unique()} athletes")
display(meta.group_by('domain', 'kind').len().sort('domain', 'kind'))

## Part 4–6 · Physical definitions, baselines (F0, random, F1) and transductive encoders

Physics targets (per sequence, from its own frames): max speed, speed gain, mean a_tan, max |a_n|, |heading change| (only with ≥ 50 % valid heading), valid duration. Matched state: start-speed bin × accelerating flag. Category map: forty / pass-rush ↔ accelerating demands, shuttle / cone / transition ↔ braking-turning demands, gauntlet ↔ sustained high speed.

In [ ]:
EVAL_T, PROBES, CURVES, EMB = [], [], [], {}
E0 = rep.engineered_features(X, M)
r, pr = se.evaluate('F0_engineered', m12c.TRANSDUCTIVE, E0, meta, PHYS, FOLD, C, C.seed)
EVAL_T.append(r)
PROBES.append(pr)
rnd = se.random_frozen(se.scale_config(se.by_name(se.PRIMARY), C), DEV, C.seed)
Er = se.embed(rnd, X, M, DEV)
r, pr = se.evaluate('random_frozen_F4_architecture', m12c.TRANSDUCTIVE, Er, meta, PHYS, FOLD, C, C.seed)
EVAL_T.append(r)
PROBES.append(pr)
CONFIGS = list(se.FAMILIES) + (list(se.ABLATIONS) if RUN_ABLATIONS else [])
if TRAIN_TRANSDUCTIVE:
    tr, va, _ = se.split_indices(meta, FOLD, None, 0.1, C.seed)
    for e in CONFIGS:
        ec = se.scale_config(e, C)
        model, curve, sig = se.train(ec, X, M, P, STATE, CAT, DOM, tr, va, C, DEV, store.models_dir, controls.run_mode, 'transductive', controls.smoke, C.seed)
        E = se.embed(model, X, M, DEV)
        EMB[e.name] = E
        rep.save_embeddings(store.data_dir, f'12f_{e.name}_transductive', E, meta, sig)
        r, pr = se.evaluate(e.name, m12c.TRANSDUCTIVE, E, meta, PHYS, FOLD, C, C.seed)
        EVAL_T.append(r)
        PROBES.append(pr)
        CURVES.append(curve.with_columns(pl.lit(e.name).alias('encoder')))
        del model
        if DEV == 'cuda':
            torch.cuda.empty_cache()
EVT = pl.DataFrame(EVAL_T, infer_schema_length=None)
PRB = pl.concat(PROBES, how='diagonal_relaxed')
CRV = pl.concat(CURVES, how='diagonal_relaxed') if CURVES else pl.DataFrame()
null = se.shuffled_identity_null(EMB.get(se.PRIMARY, E0), meta, 10 if not controls.smoke else 2, C.seed)
W(EVT, 'm12f_eval_transductive')
W(PRB, 'm12f_probes_transductive')
W(CRV, 'm12f_training_curves')
W(pl.DataFrame({'replicate': np.arange(null.size), 'cross_auc_null': null}), 'm12f_identity_null')
with pl.Config(tbl_rows=-1, tbl_width_chars=300):
    display(EVT.select([c for c in ('representation', 'training_mode', 'within_top1', 'within_auc', 'cross_auc', 'cross_auc_lo', 'cross_auc_hi', 'cross_n_athletes', 'neighbour_dspeed', 'random_dspeed',
                                    'domain_auc', 'mmd2', 'probe_r2_median') if c in EVT.columns]))
print(f'shuffled-identity null cross AUC: mean {np.nanmean(null):.3f}, max {np.nanmax(null):.3f}')

## Part 7a · Learning curve against unlabeled NFL data (transductive, F4)

In [ ]:
LC = []
if LEARNING_CURVE and TRAIN_TRANSDUCTIVE:
    tr, va, _ = se.split_indices(meta, FOLD, None, 0.1, C.seed)
    for frac in (0.1, 0.3):
        ec = se.scale_config(se.by_name(se.PRIMARY), C)
        model, curve, sig = se.train(ec, X, M, P, STATE, CAT, DOM, tr, va, C, DEV, store.models_dir, controls.run_mode, f'lc{int(frac * 100)}', controls.smoke, C.seed, nfl_fraction=frac)
        r, _ = se.evaluate(f'{se.PRIMARY}@nfl{int(frac * 100)}%', m12c.TRANSDUCTIVE, se.embed(model, X, M, DEV), meta, PHYS, FOLD, C, C.seed)
        LC.append({'nfl_fraction': frac, **r})
    full = EVT.filter(pl.col('representation') == se.PRIMARY)
    if full.height:
        LC.append({'nfl_fraction': 1.0, **full.row(0, named=True)})
LCT = pl.DataFrame(LC, infer_schema_length=None)
W(LCT, 'm12f_learning_curve')
display(LCT.select([c for c in ('nfl_fraction', 'cross_auc', 'cross_auc_lo', 'cross_auc_hi', 'probe_r2_median', 'domain_auc') if c in LCT.columns]) if LCT.height else LCT)

## Part 7b · INDUCTIVE encoders (per outer fold) — held-out retrieval and NFL transfer

For each fold f: train without fold-f athletes (both domains); evaluate cross-domain retrieval on fold-f athletes only; store every athlete's mean Combine forty embedding from that encoder (used for fold-f predictions only). Ladder on the 12C frozen target: M1 · M1 + F0 (engineered athlete means; no fitting) · M1 + F1 TCN-B (Milestone 06 fold encoders, when present) · M1 + F2 · M1 + F4 (≤ 8 training-fold PCA dims).

In [ ]:
BYF, EVAL_I = {}, []
if TRAIN_INDUCTIVE:
    folds = sorted(set(FOLD.values()))
    for name in se.INDUCTIVE_CONFIGS:
        BYF[name] = {}
        for f in folds:
            tr, va, ho = se.split_indices(meta, FOLD, f, 0.1, C.seed)
            model, curve, sig = se.train(se.scale_config(se.by_name(name), C), X, M, P, STATE, CAT, DOM, tr, va, C, DEV, store.models_dir, controls.run_mode, f'inductive_f{f}', controls.smoke, C.seed)
            E = se.embed(model, X, M, DEV)
            BYF[name][f] = se.athlete_mean_embedding(E, meta)
            W(BYF[name][f].with_columns(pl.lit(f).alias('held_out_fold')), f'm12f_inductive_{name}_f{f}')
            mh = meta[ho]
            xr = rep.cross_domain_retrieval(E[ho], mh, max(200, C.n_boot // 5), C.seed)
            EVAL_I.append({'representation': name, 'training_mode': m12c.INDUCTIVE, 'held_out_fold': f, **xr, 'checkpoint': sig.get('checkpoint_dir')})
            del model
    for f in folds:
        _, _, ho = se.split_indices(meta, FOLD, f, 0.1, C.seed)
        EVAL_I.append({'representation': 'F0_engineered', 'training_mode': m12c.INDUCTIVE, 'held_out_fold': f, **rep.cross_domain_retrieval(E0[ho], meta[ho], max(200, C.n_boot // 5), C.seed)})
EVI = pl.DataFrame(EVAL_I, infer_schema_length=None)
EVI_SUM = (EVI.group_by('representation').agg(pl.col('auc').mean().alias('cross_auc_mean_over_folds'), pl.col('auc').min().alias('min_fold_auc'), pl.col('n_athletes').sum().alias('n_athletes'))
           if EVI.height else pl.DataFrame())
W(EVI, 'm12f_eval_inductive')
W(EVI_SUM, 'm12f_eval_inductive_summary')
display(EVI_SUM)

In [ ]:
TRF = pl.DataFrame()
if BYF and store.has('m12c_targets'):
    reg = D.m12('m12_cohort_registry')
    tg = store.read_frame('m12c_targets', fd.NB['12C']).select('nfl_id', 'nfl.q90_shared')
    e0a = rep.athlete_embedding_table(E0, meta, 'combine', ['forty'], prefix='eng')
    data = (off.filter(pl.col('nfl_id').is_in(reg.filter(pl.col('linked'))['nfl_id'].to_list())).join(tg, on='nfl_id', how='inner')
            .join(e0a, on='nfl_id', how='left').join(D.m12('m12_fold_manifest').select('nfl_id', 'fold'), on='nfl_id', how='inner').sort('nfl_id'))
    mods = {'M1': ae.Model(list(ae.M1_COLS)), 'M1+F0': ae.Model([*ae.M1_COLS, *[c for c in e0a.columns if c != 'nfl_id']])}
    for name, byf in BYF.items():
        mods[f'M1+{name}'] = ae.Model(list(ae.M1_COLS), factory=lambda a, _b=byf: se.embedding_block(a, FOLD, _b))
    try:
        tcn = store.transfer.require_named('transfer_attempt_representations_outer', '06b')
        mods['M1+F1_TCN'] = ae.Model(list(ae.M1_COLS), factory=lambda a, _b=se.tcn_by_fold(tcn): se.embedding_block(a, FOLD, _b))
    except Exception as e_:
        print('F1 TCN-B representations not available:', type(e_).__name__)
    res = ae.oof_ladder(data, 'nfl.q90_shared', mods, C, C.seed)
    comps_ = [(m, 'M1') for m in mods if m != 'M1'] + [(f'M1+{n}', 'M1+F0') for n in BYF]
    TRF = ae.compare(res, comps_, C, C.seed, target='nfl.q90_shared', training_regime=m12c.INDUCTIVE)
W(TRF, 'm12f_transfer')
display(TRF.select('model', 'reference', 'n_athletes', 'delta_rmse', 'delta_lo', 'delta_hi', 'verdict') if TRF.height else Markdown('_NFL transfer not run (needs inductive encoders and 12C)_'))

## Part 8–9 · Uncertainty, ablations and figures

In [ ]:
if CRV.height:
    store.show(fg.fig_training(CRV, 'Training and validation losses', 'Validation loss (solid, held-out athletes) and training loss components (dashed) per epoch; early stopping on validation; wall-clock cap per encoder.'),
               'AM12_F_01_training_curves', {'curves': CRV}, source='m12f_training_curves')
store.show(fg.fig_heatmap(PRB, 'representation', 'probe', 'r2_cv', 'Physical probe quality', 'Athlete-grouped cross-validated R² of a ridge probe from each representation to each physical scalar (both domains pooled per probe row).', fmt='{:.2f}'),
           'AM12_F_02_physical_probes', {'probes': PRB}, publication=True, source='m12f_probes_transductive')
Ep = EMB.get(se.PRIMARY)
if Ep is not None:
    store.show(fg.fig_embedding_map(Ep, meta, 'Cross-domain embedding map (F4, transductive)', 'Two-dimensional projection of sequence embeddings coloured by domain / movement kind (a picture is not evidence; see the retrieval tables).'),
               'AM12_F_03_embedding_map', {}, publication=True, source='m12f embeddings')
ef = EVT.filter(pl.col('cross_auc').is_not_null()).with_columns((pl.col('representation') + ' (' + pl.col('training_mode') + ')').alias('label'))
store.show(fg.fig_forest(ef, 'label', 'cross_auc', 'cross_auc_lo', 'cross_auc_hi', 'Cross-domain athlete retrieval (TRANSDUCTIVE diagnostics)', f'Mean AUC of an athlete’s own NFL embedding among same-position athletes for its Combine query (0.5 = chance; shuffled-identity null mean {np.nanmean(null):.3f}). Ablations and the shuffled-physics control included.',
                         'retrieval AUC', ref=0.5), 'AM12_F_04_retrieval_transductive', {'eval': ef}, publication=True, source='m12f_eval_transductive')
dd_ = EVT.select('representation', 'domain_auc', 'mmd2').unpivot(index='representation', variable_name='metric', value_name='value')
store.show(fg.fig_heatmap(dd_, 'representation', 'metric', 'value', 'Domain-distance diagnostics', 'Domain-classifier AUC (1 = domains fully separable by a linear read-out) and RBF-MMD². Separability is NOT failure: the tasks differ.', fmt='{:.2f}'),
           'AM12_F_05_domain_distance', {'domain': dd_}, source='m12f_eval_transductive')
if EVI.height:
    ei = EVI.with_columns((pl.col('representation') + ' · fold ' + pl.col('held_out_fold').cast(pl.String)).alias('label'))
    store.show(fg.fig_forest(ei, 'label', 'auc', 'auc_lo', 'auc_hi', 'Cross-domain retrieval of held-out athletes (INDUCTIVE)', 'Encoders trained without the held-out fold’s athletes (either domain); retrieval among held-out athletes only.', 'retrieval AUC', ref=0.5, color_by='representation'),
               'AM12_F_06_retrieval_inductive', {'eval': ei}, publication=True, source='m12f_eval_inductive')
if TRF.height:
    tf = TRF.with_columns((pl.col('model') + ' vs ' + pl.col('reference') + ' (n=' + pl.col('n_athletes').cast(pl.String) + ')').alias('label'))
    store.show(fg.fig_forest(tf, 'label', 'delta_rmse', 'delta_lo', 'delta_hi', 'NFL transfer: physical features versus learned representations (INDUCTIVE)', 'Δ RMSE on the 12C frozen target, identical held-out athletes, paired athlete bootstrap.', 'Δ RMSE [yd/s²]', color_by='model'),
               'AM12_F_07_transfer', {'transfer': tf}, publication=True, source='m12f_transfer')
if LCT.height:
    store.show(fg.fig_lines(LCT, 'nfl_fraction', 'cross_auc', None, 'Learning curve versus unlabeled NFL data', 'Transductive F4 cross-domain retrieval AUC with 10 %, 30 % and 100 % of the NFL windows (same Combine data).', 'share of NFL windows', 'retrieval AUC', 'cross_auc_lo', 'cross_auc_hi', ref=0.5),
               'AM12_F_08_learning_curve', {'lc': LCT}, source='m12f_learning_curve')
if Ep is not None:
    mm = meta.with_row_index('_r')
    cmb, nfl_ = mm.filter((pl.col('domain') == 'combine') & (pl.col('kind') == 'forty')), mm.filter(pl.col('domain') == 'nfl')
    q = cmb.head(5)
    Sx = rep._cos(Ep[q['_r'].to_numpy()], Ep[nfl_['_r'].to_numpy()])
    ex = [{'query_athlete': int(a), 'neighbour_athlete': int(nfl_['nfl_id'][j]), 'neighbour_kind': nfl_['kind'][j], 'same_athlete': int(a) == int(nfl_['nfl_id'][j])}
          for a, row in zip(q['nfl_id'].to_list(), Sx) for j in np.argsort(-row)[:3]]
    display(Markdown('**Matched examples (F4, transductive):** nearest NFL windows of five Combine forty attempts.'))
    display(pl.DataFrame(ex))

## Part 10–12 · Conclusions, comparisons for 12G, artifacts, next questions

In [ ]:
rm = controls.run_mode
comps = []
base = EVT.filter(pl.col('representation') == 'F0_engineered')
for r in EVT.iter_rows(named=True):
    if r.get('cross_auc') is not None:
        comps.append(sy.comparison_row('12F', f"retrieval AUC {r['representation']} − 0.5 (transductive)", 'AUC − 0.5', r['cross_auc'] - 0.5,
                                       (r['cross_auc_lo'] - 0.5) if r.get('cross_auc_lo') is not None else None, (r['cross_auc_hi'] - 0.5) if r.get('cross_auc_hi') is not None else None,
                                       r.get('cross_n_athletes'), m12c.PHYSICAL_TRANSFER, 'm12f_eval_transductive', rm, experiment=6, training_regime=m12c.TRANSDUCTIVE))
for r in (TRF.iter_rows(named=True) if TRF.height else []):
    comps.append(sy.comparison_row('12F', f"{r['model']} vs {r['reference']} → nfl.q90_shared", 'Δ RMSE', r['delta_rmse'], r['delta_lo'], r['delta_hi'], r['n_athletes'], m12c.PHYSICAL_TRANSFER,
                                   'm12f_transfer', rm, experiment=6, training_regime=m12c.INDUCTIVE, primary=(r['model'], r['reference']) in ((f'M1+{se.PRIMARY}', 'M1'), (f'M1+{se.PRIMARY}', 'M1+F0'))))
COMP = sy.comparison_frame(comps)
W(COMP, 'm12f_comparisons')
ev_rows = []
for r in EVT.iter_rows(named=True):
    ev_rows.append(ae.evidence_row(f"12F.{r['representation']}.transductive", '12F', experiment='6', physical_interpretation='learned sequence representation (physics-probed)', input_data='sequence store',
                                   output='retrieval auc', n_athletes=r.get('cross_n_athletes'), cross_domain_stat=r.get('cross_auc'), cross_domain_lo=r.get('cross_auc_lo'), cross_domain_hi=r.get('cross_auc_hi'),
                                   reliability=r.get('within_auc'), compute_cost='seconds' if r['representation'].startswith(('F0', 'random')) else 'hours', training_regime=m12c.TRANSDUCTIVE,
                                   evaluation_target=m12c.PHYSICAL_TRANSFER, run_mode=rm, config_hash=C.hash(), limitations='transductive: label-free diagnostics only'))
for r in (TRF.filter(pl.col('reference') == 'M1').iter_rows(named=True) if TRF.height else []):
    ev_rows.append(ae.evidence_row(f"12F.{r['model']}.inductive", '12F', experiment='6', physical_interpretation='inductive fold embedding', input_data='sequence store', output='nfl.q90_shared',
                                   n_athletes=r['n_athletes'], delta_rmse=r['delta_rmse'], delta_lo=r['delta_lo'], delta_hi=r['delta_hi'], compute_cost='hours', training_regime=m12c.INDUCTIVE,
                                   evaluation_target=m12c.PHYSICAL_TRANSFER, run_mode=rm, config_hash=C.hash()))
EVID = ae.evidence_frame(ev_rows)
W(EVID, 'm12f_evidence')
prim = COMP.filter(pl.col('primary'))['verdict'].to_list()
complete = bool(TRAIN_INDUCTIVE and TRF.height)
status = rp.status_from(prim, completed=complete)
facts = {'status': status, 'run_mode': rm, 'sample': f"{X.shape[0]:,} sequences, {meta['nfl_id'].n_unique()} athletes; device {DEV}.",
         'evidence': [f"{r['representation']} (transductive): cross AUC {rp.ci(r.get('cross_auc'), r.get('cross_auc_lo'), r.get('cross_auc_hi'))}, domain AUC {rp.fmt(r.get('domain_auc'))}, probe R² median {rp.fmt(r.get('probe_r2_median'))}" for r in EVT.iter_rows(named=True)]
                     + ([f"{r['representation']} (inductive): mean held-out AUC {rp.fmt(r['cross_auc_mean_over_folds'])}" for r in EVI_SUM.iter_rows(named=True)] if EVI_SUM.height else [])
                     + [f"{r['model']} vs {r['reference']}: Δ RMSE {rp.ci(r['delta_rmse'], r['delta_lo'], r['delta_hi'], 4)} — {r['verdict']}" for r in (TRF.iter_rows(named=True) if TRF.height else [])],
         'alternatives': ['Retrieval can rise through position-typical or body-size-typical movement (retrieval is within position to limit this).',
                          'Matched-state alignment may pull domains together without athlete information (compare F4 with F4_noalign and the random encoder).'],
         'relevance': 'Tests whether large unlabeled NFL movement improves a pre-draft movement representation beyond engineered physics.',
         'limitations': ['One architecture size per family; no search.', 'Wall-clock caps can stop training early (recorded per checkpoint).', 'One cohort.'],
         'next': ['Continue only if the inductive F4 beats F0 and random on held-out retrieval AND adds NFL prediction.']}
md = rp.conclusion_markdown('12F', facts)
display(Markdown(md))
rp.write_conclusion(store, '12F', md)
rp.update_doc(store, 'SHARED_REPRESENTATIONS.md', md + chr(10) + '### Transductive diagnostics' + chr(10) + chr(10) + rp.md_table(EVT, ['representation', 'cross_auc', 'cross_auc_lo', 'cross_auc_hi', 'within_auc', 'domain_auc', 'probe_r2_median'])
              + chr(10) + chr(10) + '### Inductive (separate table)' + chr(10) + chr(10) + rp.md_table(EVI_SUM) + chr(10) + chr(10) + rp.md_table(TRF, ['model', 'reference', 'n_athletes', 'delta_rmse', 'delta_lo', 'delta_hi', 'verdict']))
rp.run_manifest(store, '12F', PROV, ['m12f_eval_transductive', 'm12f_eval_inductive', 'm12f_transfer', 'm12f_comparisons', 'm12f_evidence'], status, {'device': DEV, 'inductive': complete})
print('Send back: checks, sequence counts, the transductive table, the inductive summary, the transfer table, learning curve, the conclusion.')
print('Next: 12g_advanced_movement_research_synthesis.ipynb')